# Lab 01 — Arquitetura em camadas de um DW (DuckDB)

**Onde roda:** 🟢 Browser (JupyterLite).

Objetivo: sentir o fluxo **raw → core → mart** — dados crus, depois limpos/integrados, depois prontos para consumo.

In [ ]:
try:
    import duckdb
except ModuleNotFoundError:
    import piplite; await piplite.install('duckdb'); import duckdb
con = duckdb.connect()
# RAW: como os dados chegam da fonte — com sujeira (duplicata, estado em caixa variada)
con.execute('CREATE TABLE raw_pedidos(pedido_id INT, cliente VARCHAR, estado VARCHAR, valor DOUBLE)')
con.executemany('INSERT INTO raw_pedidos VALUES (?,?,?,?)', [
    (1,'ana','sp',100.0),(2,'bruno','RJ',200.0),(1,'ana','sp',100.0),
    (3,'caio','mg',50.0),(4,'ana','SP',80.0)])
con.execute('SELECT * FROM raw_pedidos').df()

## Camada CORE — limpar e integrar
Remove duplicatas e padroniza o estado (a 'fonte da verdade').

In [ ]:
con.execute('''CREATE TABLE core_pedidos AS
    SELECT DISTINCT pedido_id, cliente, UPPER(estado) AS estado, valor
    FROM raw_pedidos''')
con.execute('SELECT * FROM core_pedidos ORDER BY pedido_id').df()

## Camada MART — pronto para consumo
Um recorte agregado para BI: receita por estado.

In [ ]:
con.execute('''
    SELECT estado, SUM(valor) AS receita
    FROM core_pedidos
    GROUP BY estado
    ORDER BY receita DESC
''').df()

## Sua vez (mini-desafio)
A partir do **core**, traga a **receita por cliente** `(cliente, receita)`, da maior para a menor. Verifique.

In [ ]:
resposta = con.execute('''
    SELECT cliente, SUM(valor) AS receita
    FROM core_pedidos
    GROUP BY cliente
    ORDER BY receita DESC
''').fetchall()
resposta

In [ ]:
def verificar(rows):
    esperado = [('bruno',200.0),('ana',180.0),('caio',50.0)]
    try:
        assert rows == esperado, 'Agregue a partir do CORE (sem duplicatas).'
        print('✅ Correto! Você percorreu raw -> core -> mart.')
    except AssertionError as e:
        print('❌', e)

verificar(resposta)